# Module 2c - From a Live Literature Search to a Dataset

Type a topic, pull the open-access papers for it, read their **full text**, and turn them into a clean table you can model. This first part does the harvest: **an OpenAlex filter you build on the website, pasted here, becomes a corpus of full-text PDFs downloaded and parsed automatically.**

**The live-demo flow**
1. On **openalex.org** search a topic and click filters until the page reads something like:
   *works where open access is (true) and year >= (2019) and title/abstract has (battery capacity mAh) and type is (article) and citation count >= (100)*
2. Copy the filter (the site shows the API query), paste it into the config cell below.
3. Run: the notebook pulls that corpus, downloads each open-access full-text PDF, and parses it.

No key and no login for any of this (OpenAlex is free; a courtesy email is optional).

## 0. Setup

In [1]:
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "pymupdf"])
import requests, re, io, time
print("ready")

ready


## 1. Paste your OpenAlex query (OQL)

Build the query on **openalex.org** by clicking filters. The builder writes it in **OQL** (OpenAlex Query Language), e.g.

```
works where citation count >= (100)
  and open access is (true)
  and year >= (2020)
  and title/abstract has (battery capacity and mAh)
  and type is (article)
```

**Copy that OQL and paste it below** - the notebook translates it into the OpenAlex API query. (It also accepts a full `api.openalex.org/works?...` URL or a raw filter string.)

In [2]:
import re
# Paste the OQL exactly as the openalex.org builder writes it (or a URL / raw filter):
OQL = """
works where citation count >= (100)
  and open access is (true)
  and year >= (2020)
  and title/abstract has (battery capacity and mAh)
  and type is (article)
"""

USE_CACHE = True   # True: read the shipped pre-crawled cache (offline, instant). False: crawl live.
N_PAPERS = 20                       # how many full-text papers to harvest (demo: 15-30)
MAILTO   = "ruiding@uchicago.edu"   # courtesy only, NOT a key; may be ""

# OQL display-name -> OpenAlex API filter key. Names whose key is just the snake_case
# of the name are handled by the fallback below, so any extra field a student adds
# on openalex.org keeps working; only genuinely unknown fields get a warning.
OQL_FIELDS = {
    "citation count": "cited_by_count", "cited by count": "cited_by_count",
    "year": "publication_year", "publication year": "publication_year",
    "open access": "open_access.is_oa", "is open access": "open_access.is_oa",
    "title/abstract": "title_and_abstract.search", "title and abstract": "title_and_abstract.search",
    "title": "title.search", "abstract": "abstract.search", "fulltext": "fulltext.search",
    "type": "type", "language": "language", "publication date": "publication_date",
    "is retracted": "is_retracted", "has doi": "has_doi",
    "has abstract": "has_abstract", "has fulltext": "has_fulltext",
}
SEARCHY = {"title_and_abstract.search", "title.search", "abstract.search", "fulltext.search"}

def parse_oql(text):
    """Translate OpenAlex OQL into an API filter. Also accepts a full openalex.org URL or raw filter."""
    t = text.strip()
    if "openalex.org" in t and "filter=" in t:
        from urllib.parse import urlparse, parse_qs, unquote
        return unquote(parse_qs(urlparse(t).query).get("filter", [""])[0])
    if ("where" not in t.lower()) and re.search(r"[a-z_]+\.[a-z_]+:", t):
        return t
    t = re.sub(r"^\s*works\s+where\s+", "", t, flags=re.I)
    parts, guessed = [], []
    for fld, op, val in re.findall(r"([a-zA-Z /_]+?)\s*(>=|<=|>|<|is not|is|has)\s*\(([^)]*)\)", t):
        fld = re.sub(r"^and\b\s*", "", fld.strip().lower()).strip(); val = val.strip()
        key = OQL_FIELDS.get(fld)
        if not key:                                   # fallback: name -> snake_case key
            key = re.sub(r"[ /]+", "_", fld); guessed.append(f"{fld} -> {key}")
        if key in SEARCHY:
            parts.append(f"{key}:" + re.sub(r"\s+and\s+", " ", val, flags=re.I)); continue
        v = val.lower() if val.lower() in ("true", "false") else val
        neg = "!" if op == "is not" else ""
        if op == ">=":   parts.append(f"{key}:>{int(val)-1}")
        elif op == "<=": parts.append(f"{key}:<{int(val)+1}")
        elif op == ">":  parts.append(f"{key}:>{val}")
        elif op == "<":  parts.append(f"{key}:<{val}")
        else:            parts.append(f"{key}:{neg}{v}")
    if guessed:
        print("  note: guessed these field keys (check OpenAlex docs if a query errors):", guessed)
    return ",".join(parts)

FILTER = parse_oql(OQL)
print("translated OQL -> OpenAlex filter:\n ", FILTER)

translated OQL -> OpenAlex filter:
  cited_by_count:>99,open_access.is_oa:true,publication_year:>2019,title_and_abstract.search:battery capacity mAh,type:article


## 2. Pull the corpus

Ask OpenAlex for works matching the filter, sorted by citations, and keep the ones that actually have a downloadable open-access PDF.

In [3]:
import pandas as pd, gzip, json, os
CACHE = "cache"
def openalex_count(filt, mail=""):
    p={"filter":filt,"per-page":1}
    if mail: p["mailto"]=mail
    return requests.get("https://api.openalex.org/works", params=p, timeout=45).json()["meta"]["count"]
def get_corpus(filt, n, mail=""):
    base,out,cur="https://api.openalex.org/works",[], "*"
    while len(out)<n:
        p={"filter":filt,"sort":"cited_by_count:desc","per-page":min(200,n-len(out)),"cursor":cur}
        if mail: p["mailto"]=mail
        j=requests.get(base,params=p,timeout=60).json(); res=j.get("results",[])
        if not res: break
        for w in res:
            loc=w.get("best_oa_location") or {}
            out.append(dict(title=(w.get("title") or "")[:140],year=w.get("publication_year"),
                venue=(loc.get("source") or {}).get("display_name"),cited=w.get("cited_by_count"),
                doi=(w.get("doi") or "").replace("https://doi.org/",""),pdf=loc.get("pdf_url") or ""))
            if len(out)>=n: break
        cur=j.get("meta",{}).get("next_cursor")
        if not cur: break
        time.sleep(0.2)
    return out

N_LIST = 200     # size of the LIVE paper list (ignored in cache mode)
if USE_CACHE and os.path.exists(f"{CACHE}/corpus_papers.csv"):
    papers = pd.read_csv(f"{CACHE}/corpus_papers.csv").fillna("")
    corpus = papers.to_dict("records")
    n_pdf  = sum(1 for p in corpus if p["pdf"])
    print(f"[cache] loaded {len(corpus)} papers from the shipped pre-crawl ({n_pdf} with an OA PDF). Offline, instant.")
else:
    TOTAL = openalex_count(FILTER, MAILTO)
    print(f"your query matches {TOTAL:,} papers in OpenAlex.\n")
    corpus = get_corpus(FILTER, min(N_LIST, TOTAL), MAILTO)
    n_pdf  = sum(1 for p in corpus if p["pdf"])
    print(f"built a list of {len(corpus)} papers (of {TOTAL:,} total); {n_pdf} have a downloadable OA PDF.")
    papers = pd.DataFrame(corpus)
papers.head(10)

[cache] loaded 540 papers from the shipped pre-crawl (397 with an OA PDF). Offline, instant.


,title,year,venue,cited,doi,pdf
0,Fluorinated interphase enables reversible aque...,2021,OSTI OAI (U.S. Department of Energy Office of ...,1177,10.1038/s41565-021-00905-4,
1,Electrolyte design for LiF-rich solid–electrol...,2020,OSTI OAI (U.S. Department of Energy Office of ...,1138,10.1038/s41560-020-0601-1,
2,Enhanced Potassium-Ion Storage of the 3D Carbo...,2020,Nano-Micro Letters,894,10.1007/s40820-020-00525-y,https://link.springer.com/content/pdf/10.1007/...
3,Tailoring electrolyte solvation for Li metal b...,2021,PubMed Central,849,10.1038/s41560-021-00783-z,
4,Revealing the closed pore formation of waste w...,2023,Nature Communications,809,10.1038/s41467-023-39637-5,https://www.nature.com/articles/s41467-023-396...
5,Electrolyte Design for In Situ Construction of...,2021,Figshare,796,10.1002/adma.202007416,
6,"High areal capacity, long cycle life 4 V ceram...",2022,OSTI OAI (U.S. Department of Energy Office of ...,628,10.1038/s41560-021-00952-0,
7,An Inorganic‐Rich Solid Electrolyte Interphase...,2020,OSTI OAI (U.S. Department of Energy Office of ...,607,10.1002/anie.202012005,
8,Steric Effect Tuned Ion Solvation Enabling Sta...,2021,OSTI OAI (U.S. Department of Energy Office of ...,596,10.1021/jacs.1c09006,https://www.osti.gov/servlets/purl/1878520
9,MgO‐Template Synthesis of Extremely High Capac...,2020,Angewandte Chemie International Edition,579,10.1002/anie.202013951,https://onlinelibrary.wiley.com/doi/pdfdirect/...


### Download the paper list

Save the matching-paper list and pull it to your computer with one click (a browser download dialog on Colab).

In [4]:
# save the paper list and offer a one-click download to your computer
papers.to_csv("corpus_papers.csv", index=False)
papers.to_json("corpus_papers.json", orient="records", force_ascii=False, indent=1)
print("saved: corpus_papers.csv  /  corpus_papers.json")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("corpus_papers.csv")     # <- this pops a browser "save file" dialog
    # files.download("corpus_papers.json")  # uncomment to grab the JSON too
else:
    print("(on Colab this pops a download dialog; locally the files are just written next to the notebook)")

saved: corpus_papers.csv  /  corpus_papers.json
(on Colab this pops a download dialog; locally the files are just written next to the notebook)


## 3. Download and parse the full text

Each open-access PDF is downloaded (with a normal browser header) and parsed to text. A publisher that blocks automated download (some return HTTP 403) is skipped with a note; the rest go through.

In [5]:
import gzip, json, os
N_FULLTEXT = 20   # LIVE only: how many PDFs to download+parse (the slow step). Cache mode loads all pre-crawled.
if USE_CACHE and os.path.exists(f"{CACHE}/fulltext.jsonl.gz"):
    texts=[json.loads(l) for l in gzip.open(f"{CACHE}/fulltext.jsonl.gz","rt")]
    print(f"[cache] loaded full text for {len(texts)} papers from the shipped pre-crawl. Offline, instant.")
else:
    import fitz
    UA={"User-Agent":"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"}
    def full_text(u):
        r=requests.get(u,headers=UA,timeout=60)
        if r.status_code!=200 or not (r.content[:4]==b"%PDF" or "pdf" in r.headers.get("content-type","")): return None
        return "\n".join(pg.get_text() for pg in fitz.open(stream=r.content,filetype="pdf"))[:60000]
    todo=[p for p in corpus if p["pdf"]][:N_FULLTEXT]
    print(f"downloading full text for {len(todo)} papers (raise N_FULLTEXT for more; PDFs are the slow part)\n")
    texts=[]
    for p in todo:
        try: t=full_text(p["pdf"])
        except Exception: t=None
        if t: texts.append({**p,"nchars":len(t),"text":t}); print(f"  ok  {len(t):6d} chars  {str(p['venue'])[:34]}")
        else: print(f"  skip (blocked/not a PDF)   {str(p['venue'])[:34]}")
        time.sleep(0.4)
    print(f"\nfull text obtained for {len(texts)}/{len(todo)} papers")
print(f"\n-> {len(texts)} full-text papers ready for extraction (next part).")

[cache] loaded full text for 139 papers from the shipped pre-crawl. Offline, instant.

-> 139 full-text papers ready for extraction (next part).


## Next

Part 2 (coming next) sends each full text through the Module 2a extractor to pull a structured record (material, chemistry, specific capacity, rate, cycles) into `battery_dataset.csv`, then feeds that table into Module 1d to model it. This notebook already gives you the working harvest: **topic filter -> corpus -> parsed full text**, with no key.